In [1]:
import pyodbc
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

In [2]:
server = 'sharktowels.duckdns.org'
database = "Spending"
user = "SA"

with open("password.txt", "r") as file:
    password = file.read().strip()

connection_string = (
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={user};"
    f"PWD={password};"
    "TrustServerCertificate=yes;"
)

conn = pyodbc.connect(connection_string)


In [ ]:
user = 12


C:\Users\jduen\AppData\Local\Temp\ipykernel_16376\2537624565.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  area_df = pd.read_sql_query(f'''


Category,entertainment,food,future,health,housing,insurance,miscellaneous,supplies,transportation,utilities
TimeDate,,,,,,,,,,
2023-12-22 02:55:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,235.12
2024-01-19 02:55:00,NaN,NaN,NaN,165.05,NaN,NaN,NaN,NaN,NaN,235.12
2024-03-16 02:55:00,NaN,NaN,NaN,165.05,NaN,NaN,17.65,NaN,NaN,235.12
2024-03-29 02:55:00,NaN,NaN,NaN,165.05,NaN,NaN,273.16,NaN,NaN,235.12
2024-04-13 02:55:00,NaN,NaN,103.07,165.05,NaN,NaN,273.16,NaN,NaN,235.12


In [ ]:
area_df = pd.read_sql_query(f'''
                            SELECT Purchases.TimeDate, Purchases.Category, Purchases.Subcategory, Payments.Amount 
                            FROM Purchases
                            JOIN Payments ON Purchases.PaymentID = Payments.PaymentID
                            WHERE UserID = {user}
                            ORDER BY Purchases.TimeDate
                            ''', conn)

area_df["runningAmount"] = area_df.sort_values("TimeDate") \
                         .groupby("Category")["Amount"] \
                         .cumsum()

pivoted_area_df = area_df.pivot_table(
    index="TimeDate",
    columns="Category",
    values="runningAmount",
    aggfunc="last"
)

pivoted_area_df = pivoted_area_df.ffill()

pivoted_area_df.head()

fig = go.Figure()

for category in pivoted_area_df.columns:
    fig.add_trace(
        go.Scatter(
            x=pivoted_area_df.index,
            y=pivoted_area_df[category],
            mode="lines",
            stackgroup="one",
            name=category
        )
    )

fig.update_layout(
    title="Cumulative spending by category",
    xaxis_title="Time",
    yaxis_title="Running Amount",
    hovermode="x unified",
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                dict(count=1,
                     label="1m",
                     step="month",
                     stepmode="backward"),
                dict(count=6,
                     label="6m",
                     step="month",
                     stepmode="backward"),
                dict(count=1,
                     label="YTD",
                     step="year",
                     stepmode="todate"),
                dict(count=1,
                     label="1y",
                     step="year",
                     stepmode="backward"),
                dict(step="all")
            ])
        ),
    )
)

fig.show()

In [29]:
pie_df = pd.read_sql_query(f'''
                            SELECT Purchases.Category, Purchases.Subcategory, Payments.Amount
                            FROM Purchases
                            JOIN Payments ON Purchases.PaymentID = Payments.PaymentID
                            WHERE UserID = {user}
                            ORDER BY Purchases.TimeDate
                            ''', conn)

pie_df.head()

categories = pie_df['Category'].unique()
subcategories = pie_df['Subcategory'].unique()

grouped_count = pie_df.groupby(["Category", "Subcategory"]).size().reset_index(name="count")

categories = grouped_count["Category"].unique()

fig = go.Figure()

for i, category in enumerate(categories):
    category_df = grouped_count[grouped_count["Category"] == category]

    fig.add_trace(

        go.Pie(
            labels = category_df["Subcategory"],
            values = category_df["count"],
            name = category,
            visible = (i == 0),
            hovertemplate = "%{label}, %{value}<extra></extra>"
        )

    )

fig.update_layout(

    updatemenus=[

        dict(
            buttons = [
                dict(
                    label = category,
                    method = "update",
                    args = [
                        {"visible": [j == i for j in range(len(categories))]},
                        {"title": f"Number of Purchases in: {category}"}
                    ]
                )
                for i, category in enumerate(categories)
            ],
            direction = "down",
            x = 1,
            y = 1,
        )

    ],
    title = f"Number of Purchases in: {categories[0]}"
)

fig.show()


C:\Users\jduen\AppData\Local\Temp\ipykernel_16376\1024574105.py:1: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



In [37]:
grouped_amount = pie_df.groupby(["Category", "Subcategory"])["Amount"].sum().reset_index(name="total")

categories = grouped_amount["Category"].unique()

fig = go.Figure()

for i, category in enumerate(categories):
    category_df = grouped_amount[grouped_amount["Category"] == category]

    fig.add_trace(

        go.Pie(
            labels = category_df["Subcategory"],
            values = category_df["total"],
            name = category,
            visible = (i == 0),
            hovertemplate = "%{label}: $%{value}<extra></extra>"
        )

    )

fig.update_layout(

    updatemenus=[

        dict(
            buttons = [
                dict(
                    label = category,
                    method = "update",
                    args = [
                        {"visible": [j == i for j in range(len(categories))]},
                        {"title": f"Number of Purchases in: {category}"}
                    ]
                )
                for i, category in enumerate(categories)
            ],
            direction = "down",
            x = 1,
            y = 1,
        )

    ],
    title = f"Number of Purchases in: {categories[0]}"
)

fig.show()